# Lunar Orbit Mock Data and Linear Inversion

Simulate auto-correlations for two dipole antennas mounted on a torque-free
tumbling spacecraft in a 150 km equatorial lunar orbit.  Then perform a
linear inversion (Tikhonov regularisation) to jointly recover:

- per-pixel GSM brightness temperature (nside = 8)
- lunar regolith temperature  T_reg  (scalar)
- sun effective temperature  T_sun  (scalar)

The observation equation is

$$T_\text{ant}^{(i,d)} = \frac{\displaystyle\sum_j B_d(\hat n_j,t_i)\,T_\text{sky}^{(i)}(j) + T_\text{sun}\,B_d(\hat n_{\odot,i},t_i)\,m_i(j_{\odot,i})}{\displaystyle\sum_j B_d(\hat n_j,t_i)}$$

where

$$B_d(\hat n) = \left[\frac{\cos(k h_d\,\hat d\cdot\hat n) - \cos(k h_d)}{\sqrt{1-(\hat d\cdot\hat n)^2}}\right]^2$$

is the frequency-dependent center-fed dipole power pattern,
$k = 2\pi f/c$ is the wavenumber, $h_d = L_d/2$ is the half-length of dipole $d$,
$m_i(j)\in\{0,1\}$ is the lunar-occultation mask, and
$T_\text{sky}^{(i)}(j)=T_\text{GSM}(j)\,m_i(j)+T_\text{reg}\,(1-m_i(j))$.

The on-axis limit ($\hat d\cdot\hat n \to \pm 1$, $\sin\theta \to 0$) is zero for
all $kh_d$ (numerator vanishes first by L'Hôpital).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import healpy
from scipy.spatial.transform import Rotation
from astropy.time import Time
from astropy.coordinates import get_body
import astropy.units as u

from eigsep_sim.lunar_orbit import LunarOrbit, OrbiterMission
from eigsep_sim.const import R_MOON
from eigsep_sim.beam import thin_dipole_pattern
from eigsep_sim.sky import SkyModel
from eigsep_sim.sim import compute_masks_and_beams, simulate_observations
from eigsep_sim.linear_solver import build_design_matrix, svd_solve

## Configuration

In [ ]:
_YAML = os.path.join(os.path.dirname(os.path.abspath('__file__')), "bloom_config.yaml")
cfg = OrbiterMission(_YAML)

print(cfg)
print(f"Orbits: {cfg.observation.n_orbits},  Pixels: {cfg.observation.npix},  "
      f"Observations: {cfg.observation.n_rows} ({cfg.observation.n_orbits} orbits × {cfg.observation.n_obs} times × 2 dipoles)")
for o, v in enumerate(cfg.observation.rot_orbit_vecs):
    print(f"  orbit {o} normal (galactic): {v.round(4)}")
print(f"Arm lengths: {cfg.antenna.arm_lengths} m,  masses: {cfg.antenna.arm_masses} kg,  "
      f"opening angle: {cfg.antenna.opening_angle_deg}°")
print(f"Attitude model: {'fixed spin around cfg.antenna.l_hat' if cfg.observation.fixed_spin else 'ergodic tumbling (random SO(3))'}")

## Spacecraft orientations

Two attitude models are supported, selected by `cfg.observation.fixed_spin` in
`bloom_config.yaml`.

**Ergodic tumbling** (`cfg.observation.fixed_spin = False`, default):
The L and I of the spinner are designed so that the torque-free precession
is ergodic — the body visits all orientations with roughly equal probability
over a month.  We model the attitude at each observation epoch as an
independent draw from the uniform (Haar) measure on SO(3), which is both the
correct limiting distribution and orders of magnitude cheaper than integrating
the full rigid-body ODE.  A separate set of `cfg.observation.n_obs` draws is generated for
each orbital plane with independent random seeds.

**Fixed spin** (`cfg.observation.fixed_spin = True`):
The spacecraft is spin-stabilised: the body rotates uniformly around
`cfg.antenna.l_hat`, treated as a fixed axis in galactic Cartesian coordinates.  The
phase advances uniformly over `[0, 2π)` across the `cfg.observation.n_obs` observations
so that exactly one full rotation is sampled per orbit.  All orbital planes
share the same phase grid (the spin axis is inertially fixed).

In [ ]:
print(f"Dipole body-frame axes:\n  dipole 0: {cfg.antenna.u_body[0]}\n  dipole 1: {cfg.antenna.u_body[1]}")

if cfg.observation.fixed_spin:
    # Spin-stabilised: body rotates uniformly around cfg.antenna.l_hat (galactic frame).
    # cfg.antenna.l_hat is already normalised by OrbiterMission.
    phi = np.linspace(0.0, 2.0 * np.pi, cfg.observation.n_obs, endpoint=False)
    rot_fixed = Rotation.from_rotvec(np.outer(phi, cfg.antenna.l_hat))
    rots_per_orbit = [rot_fixed for _ in range(cfg.observation.n_orbits)]
    print(f"Fixed spin axis (galactic): cfg.antenna.l_hat = {cfg.antenna.l_hat.round(4)}")
    print(f"Uniform phase grid: {cfg.observation.n_obs} steps over [0, 2π) × {cfg.observation.n_orbits} orbits")
else:
    # Sample cfg.observation.n_obs orientations per orbit independently from SO(3) — the Haar measure.
    # Each orbit gets a different seed so its attitude history is statistically independent.
    rots_per_orbit = [Rotation.random(cfg.observation.n_obs, random_state=42 + o) for o in range(cfg.observation.n_orbits)]
    print(f"Sampled {cfg.observation.n_obs} SO(3) orientations × {cfg.observation.n_orbits} orbits")

## Orbital setup, GSM, and Sun positions

In [ ]:
# Create one LunarOrbit per orbital plane from the Observation config.
orbits_list = cfg.observation.make_orbits(rot_spin_vec=(0, 0, 1), spin_period=0.0)

# Shared observation time grid — all orbits cover the same window.
# (Sun direction depends only on Earth's position, not the spacecraft orbit.)
t_obs_s   = np.linspace(0.0, cfg.observation.n_days * 86400.0, cfg.observation.n_obs, endpoint=False)
obs_times = cfg.observation.obs_epoch + t_obs_s * u.s

# GSM sky at cfg.observation.freq, downsampled to cfg.observation.nside
sky     = SkyModel(np.array([cfg.observation.freq]), nside=cfg.observation.nside, srcs=None)
gsm_map = np.asarray(sky.map).squeeze().astype(float)   # (npix,) — squeeze handles 1-freq case
print(f"GSM at {cfg.observation.freq_mhz:.0f} MHz (nside={cfg.observation.nside}): "
      f"min={gsm_map.min():.0f} K, mean={gsm_map.mean():.0f} K, max={gsm_map.max():.0f} K")

# Galactic pixel unit vectors  (3, npix)
N_GAL = np.array(healpy.pix2vec(cfg.observation.nside, np.arange(cfg.observation.npix)))  # (3, npix)

# Sun positions — identical for all orbits since they share the same time grid
print("Querying Sun positions … ", end="", flush=True)
sun_coords = get_body("sun", obs_times)
sun_gal    = sun_coords.galactic
l_s = sun_gal.l.rad
b_s = sun_gal.b.rad
N_SUN = np.array([
    np.cos(b_s) * np.cos(l_s),
    np.cos(b_s) * np.sin(l_s),
    np.sin(b_s),
])
J_SUN = healpy.vec2pix(cfg.observation.nside, *N_SUN)   # (cfg.observation.n_obs,) — shared across orbits
print("done")

# Per-dipole thermal noise via the radiometer equation:
#   σ_d = T_sys_d / sqrt(Δν · τ),   T_sys_d = η_d · T_gsm_avg + T_rx
# Uses the actual GSM mean at cfg.observation.freq for T_gsm_avg.
T_gsm_avg   = float(gsm_map.mean())
SIGMA_NOISE = cfg.sigma_noise(t_gsm_avg=T_gsm_avg)
print(f"\nRadiometer noise (Δν={cfg.observation.channel_width_khz:.2f} kHz, τ={cfg.observation.t_integration} s, "
      f"T_gsm_avg={T_gsm_avg:.0f} K):")
for d, (L, sigma) in enumerate(zip(cfg.antenna.arm_lengths, SIGMA_NOISE)):
    print(f"  dipole {d} (L={L} m): σ_noise={sigma:.2f} K")

## Precompute per-observation quantities

For each (orbit, time) pair we need:
- lunar occultation mask  $m_i(j)$  from `LunarOrbit.above_horizon`  — differs per orbit
- dipole axes in galactic frame from the spinner quaternion
- dipole beam patterns  $B_0, B_1$  over all sky pixels

All results are stored in flat arrays indexed by  $k = o \cdot N_\text{obs} + i$.

In [ ]:
print(f"Dipole electrical half-lengths  kh = {cfg.kh.round(4)} rad  "
      f"({np.round(cfg.kh/np.pi, 3)} × π;  λ/2 would be π/2 = {np.pi/2:.3f})")

print("Precomputing masks and beams …")
masks, beams, omega_B = compute_masks_and_beams(
    orbits_list, obs_times, rots_per_orbit,
    cfg.antenna.u_body, cfg.kh, cfg.observation.nside,
)
print(f"Mean beam solid angles (pixels): "
      f"dipole 0 = {omega_B[:, 0].mean():.1f},  dipole 1 = {omega_B[:, 1].mean():.1f}")

## Forward model — generate mock observations

$$T_\text{ant}^{(i,d)} = \frac{\sum_j B_d^{(i)}(j)\,T_\text{sky}^{(i)}(j) + T_\text{sun}\,B_d^{(i)}(j_{\odot,i})\,m_i(j_{\odot,i})}{\sum_j B_d^{(i)}(j)}$$

In [ ]:
rng = np.random.default_rng(42)
data, y = simulate_observations(
    masks, beams, omega_B, gsm_map,
    cfg.observation.t_regolith, cfg.observation.t_sun,
    J_SUN, SIGMA_NOISE, rng=rng,
)
print(f"T_ant range: {y.min():.1f} – {y.max():.1f} K  (mean {y.mean():.1f} K)")

## Design matrix — per-pixel parameterisation

We solve directly for the temperature of each HEALPix pixel plus two scalars.
The observation row for dipole $d$ at time $i$ is

$$A[r,\,j] = \frac{B_d^{(i)}(j)\,m_i(j)}{\Omega_B^{(i,d)}}, \qquad
  A[r,\,N_\text{pix}] = \frac{\sum_j B_d^{(i)}(j)(1-m_i(j))}{\Omega_B^{(i,d)}}, \qquad
  A[r,\,N_\text{pix}+1] = \frac{B_d^{(i)}(j_{\odot,i})\,m_i(j_{\odot,i})}{\Omega_B^{(i,d)}}$$

The forward model is exact: the only residual in $\mathbf{A}\mathbf{x}_\text{true}-\mathbf{y}$ is thermal noise.

In [ ]:
N_UNKNOWNS = cfg.observation.npix + 2   # per-pixel GSM + T_reg + T_sun

A = build_design_matrix(masks, beams, omega_B, J_SUN, cfg.observation.npix)

# Truth vector — forward model is exact so residual is purely noise
x_true   = np.concatenate([gsm_map, [cfg.observation.t_regolith, cfg.observation.t_sun]])
residual = np.abs(A @ x_true - y).max()
print(f"Max |A @ x_true - y| = {residual:.2f} K  (should be ~noise σ={SIGMA_NOISE} K)")

# Single thin SVD — reused for rank/condition diagnostics, inversion, and eigenmode plots
print("Solving via thin SVD … ", end="", flush=True)
result = svd_solve(A, y, cfg.observation.npix)
U, sv, Vt = result["U"], result["sv"], result["Vt"]
print("done")

rank = result["rank"]
print(f"Shape: {A.shape},  rank: {rank} / {N_UNKNOWNS}")
print(f"Condition number: {sv[0]/sv[-1]:.2e}")

## Linear inversion

The `rcond` threshold zeros
out the null singular values corresponding to sky pixels that are
never in the open-sky footprint across all observations (always occluded
by the Moon or never illuminated with non-negligible beam weight).
Those pixels are flagged as unobserved in the output map.

In [ ]:
RCOND      = 1e-6   # matches default in svd_solve
rank_lstsq = result["rank"]

T_gsm_est = result["sky_map"].copy()
T_reg_est = result["t_regolith"]
T_sun_est = result["t_sun"]
unobserved = result["unobserved"]

T_gsm_est[unobserved] = np.nan

print(f"Unobserved pixels: {unobserved.sum()} / {cfg.observation.npix}")
obs = ~unobserved
print("Recovered scalars:")
print(f"  T_regolith : truth = {cfg.observation.t_regolith:.1f} K   est = {T_reg_est:.1f} K")
print(f"  T_sun      : truth = {cfg.observation.t_sun:.1f} K  est = {T_sun_est:.1f} K")
print(f"GSM monopole: truth = {np.mean(gsm_map[obs])} K  est = {np.mean(T_gsm_est[obs])} K")
print(f"GSM rms (observed pixels): {np.std(T_gsm_est[obs] - gsm_map[obs]):.1f} K")

## Singular value spectrum and sky eigenmodes

The thin SVD $A = U\,\Sigma\,V^T$ is computed once after building the design
matrix.  The right singular vectors $V[:,k]$ are the eigenmodes of $A^T A$
(eigenvalue $\sigma_k^2$); they show which sky directions the data can
constrain.  The same factorisation is used directly for the least-squares
inversion, so no matrix is inverted or decomposed more than once.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(sv / sv[0], ".-", ms=4)
ax.axhline(RCOND, color="r", ls="--", label="rcond threshold")
ax.set_xlabel("singular value index")
ax.set_ylabel("normalised singular value")
ax.set_title(f"Singular value spectrum  (rank {rank_lstsq} / {N_UNKNOWNS})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Vt[k, :cfg.observation.npix] reshaped as a HEALPix map is the k-th right singular vector of A,
# i.e. the k-th eigenmode of A^T A.  sv[k]^2 is the corresponding eigenvalue.
n_show    = 6
best_idx  = list(range(n_show))
worst_idx = list(range(rank_lstsq - n_show, rank_lstsq))

fig = plt.figure(figsize=(14, 6))
for col, k in enumerate(best_idx):
    healpy.mollview(
        Vt[k, :cfg.observation.npix],
        title=f"Best mode {k}  sv={sv[k]:.2e}",
        fig=fig.number, sub=(2, n_show, col + 1),
        coord="G", cmap="RdBu_r",
    )
for col, k in enumerate(worst_idx):
    healpy.mollview(
        Vt[k, :cfg.observation.npix],
        title=f"Worst mode {k}  sv={sv[k]:.2e}",
        fig=fig.number, sub=(2, n_show, n_show + col + 1),
        coord="G", cmap="RdBu_r",
    )
plt.suptitle("Best (top) and worst observed (bottom) constrained sky modes", y=1.01)
plt.tight_layout()
plt.show()

## Sky coverage over 30 days

In [ ]:
## Sky coverage: sum of (B * mask / OmB) over all observations — shows which
## galactic pixels were most frequently observed with open-sky beam weight.
#coverage = np.zeros(cfg.observation.npix)
#for o in range(cfg.observation.n_orbits):
#    for i in range(cfg.observation.n_obs):
#        k = o * cfg.observation.n_obs + i
#        m = masks[k].astype(np.float64)
#        for d in range(2):
#            coverage += beams[k, d] * m / omega_B[k, d]
#
#healpy.mollview(
#    coverage,
#    title="Sky coverage (Σ beam×mask weights over all observations)",
#    unit="a.u.", cmap="viridis", coord="G",
#)
#plt.show()

## Sky map comparison

In [ ]:
T_lim   = (gsm_map.min(), gsm_map.max())
res_lim = np.nanpercentile(np.abs(T_gsm_est - gsm_map), 95)

for title, m, kw in [
    ("Truth GSM (full resolution)",  gsm_map,
     dict(min=T_lim[0], max=T_lim[1])),
    ("Recovered GSM",                T_gsm_est,
     dict(min=T_lim[0], max=T_lim[1])),
    ("Residual (recovered − truth)", T_gsm_est - gsm_map,
     dict(min=-res_lim, max=res_lim, cmap="RdBu_r")),
]:
    healpy.mollview(m, title=title, unit="K", coord="G", **kw)
    plt.show()

## Residuals and per-pixel error

In [ ]:
## Per-pixel observation sensitivity: column norm of A gives the RMS beam weight
## each sky pixel received across all observations.
#col_norm_map = np.linalg.norm(A[:, :cfg.observation.npix], axis=0)
#
#healpy.mollview(
#    col_norm_map,
#    title="Per-pixel observation sensitivity  (‖A[:,j]‖)",
#    unit="a.u.", cmap="viridis", coord="G",
#)
#plt.show()

## Dipole beam orientations sampled over the mission

Plot the evolution of both dipole axes in galactic coordinates to verify
the torque-free tumbling provides diverse angular sampling.

In [ ]:
# Dipole axes in galactic frame at each observation time  (cfg.observation.n_obs, 2, 3)
# Show orbit 0; change the index to inspect a different orbital plane.
rots_obs  = rots_per_orbit[0]
D_gal_all = np.array([rots_obs[i].apply(cfg.antenna.u_body) for i in range(cfg.observation.n_obs)])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for d, ax in enumerate(axes):
    dx, dy, dz = D_gal_all[:, d, :].T
    sc = ax.scatter(np.arctan2(dy, dx) * 180/np.pi,
                    np.arcsin(dz)      * 180/np.pi,
                    c=t_obs_s / 86400, cmap="viridis", s=5)
    plt.colorbar(sc, ax=ax, label="day")
    ax.set_xlabel("galactic longitude [°]")
    ax.set_ylabel("galactic latitude [°]")
    ax.set_title(f"Dipole {d} axis track (galactic, orbit 0)")

plt.tight_layout()
plt.show()